# Next-pass dataset audit

Run this first. It checks dataset variables, coverage, and forecast/truth lead alignment.


In [9]:
from pathlib import Path
import sys
import pandas as pd

def detect_bundle_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd.parent.parent]
    for c in candidates:
        if (c / "src" / "flagship_predictability").exists() and (c / "examples" / "default_settings.py").exists():
            return c
    raise RuntimeError(
        "Could not detect the next-pass bundle root. "
        "Open this notebook from inside the bundle's notebooks/ folder, "
        "or manually set ROOT below."
    )

ROOT = detect_bundle_root()
SRC = ROOT / "src"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("Bundle root:", ROOT)

Bundle root: /global/u2/s/suryact/Chaos_Theory/project1_code/flagship_predictability_nextpass


In [10]:
from examples.default_settings import SETTINGS
from flagship_predictability import run_dataset_audit

SETTINGS

AtlasConfig(truth_dataset='era5_truth_240', deterministic_models={'HRES': 'hres_0012_240', 'GraphCast': 'graphcast_2020_240', 'NeuralGCM': 'neuralgcm_det_2020_240'}, ensemble_models={'IFS-ENS': 'ifs_ens_240'}, date_windows=[('2020-01-01', '2020-03-31')], leads_hours=[24, 72, 120, 168], variables={'z500': VariableSpec(name='z500', candidates=['z', 'geopotential', 'gh'], level=500, climatology_groupby=None, thresholds=[]), 't850': VariableSpec(name='t850', candidates=['t', 'temperature'], level=850, climatology_groupby=None, thresholds=[]), 'u850': VariableSpec(name='u850', candidates=['u', 'u_component_of_wind'], level=850, climatology_groupby=None, thresholds=[]), 'v850': VariableSpec(name='v850', candidates=['v', 'v_component_of_wind'], level=850, climatology_groupby=None, thresholds=[]), 'mslp': VariableSpec(name='mslp', candidates=['msl', 'mean_sea_level_pressure', 'mslp'], level=None, climatology_groupby=None, thresholds=[])}, n_regimes=4, n_eof_components=8, bootstrap_n=400, boots

In [11]:
out = run_dataset_audit(SETTINGS)
out

{'dataset_variables': PosixPath('/global/u2/s/suryact/Chaos_Theory/project1_code/flagship_predictability_nextpass/notebooks/outputs/audit/dataset_variables.csv'),
 'coverage_summary': PosixPath('/global/u2/s/suryact/Chaos_Theory/project1_code/flagship_predictability_nextpass/notebooks/outputs/audit/coverage_summary.csv'),
 'alignment_audit': PosixPath('/global/u2/s/suryact/Chaos_Theory/project1_code/flagship_predictability_nextpass/notebooks/outputs/audit/alignment_audit.csv')}

In [12]:
# Optional quick preview of CSV outputs
from pathlib import Path
for key, path in out.items():
    path = Path(path)
    print(f"\n--- {key} ---")
    print(path)
    if path.suffix.lower() == ".csv" and path.exists():
        try:
            display(pd.read_csv(path).head(10))
        except Exception as e:
            print("Could not preview CSV:", e)


--- dataset_variables ---
/global/u2/s/suryact/Chaos_Theory/project1_code/flagship_predictability_nextpass/notebooks/outputs/audit/dataset_variables.csv


,dataset,variable,dtype,dims,shape,lat_dim,lon_dim,time_dim,lead_dim,level_dim,member_dim
0,era5_truth_240,10m_u_component_of_wind,float32,"time,longitude,latitude",93544x240x121,latitude,longitude,time,NaN,NaN,NaN
1,era5_truth_240,10m_v_component_of_wind,float32,"time,longitude,latitude",93544x240x121,latitude,longitude,time,NaN,NaN,NaN
2,era5_truth_240,10m_wind_speed,float32,"time,longitude,latitude",93544x240x121,latitude,longitude,time,NaN,NaN,NaN
3,era5_truth_240,2m_dewpoint_temperature,float32,"time,longitude,latitude",93544x240x121,latitude,longitude,time,NaN,NaN,NaN
4,era5_truth_240,2m_temperature,float32,"time,longitude,latitude",93544x240x121,latitude,longitude,time,NaN,NaN,NaN
5,era5_truth_240,above_ground,float32,"time,level,longitude,latitude",93544x13x240x121,latitude,longitude,time,NaN,level,NaN
6,era5_truth_240,ageostrophic_wind_speed,float32,"time,level,longitude,latitude",93544x13x240x121,latitude,longitude,time,NaN,level,NaN
7,era5_truth_240,angle_of_sub_gridscale_orography,float32,"longitude,latitude",240x121,latitude,longitude,NaN,NaN,NaN,NaN
8,era5_truth_240,anisotropy_of_sub_gridscale_orography,float32,"longitude,latitude",240x121,latitude,longitude,NaN,NaN,NaN,NaN
9,era5_truth_240,boundary_layer_height,float32,"time,longitude,latitude",93544x240x121,latitude,longitude,time,NaN,NaN,NaN



--- coverage_summary ---
/global/u2/s/suryact/Chaos_Theory/project1_code/flagship_predictability_nextpass/notebooks/outputs/audit/coverage_summary.csv


,dataset,n_vars,lat_dim,lon_dim,time_dim,lead_dim,level_dim,member_dim,time_start,time_end,n_times,n_levels,n_leads,n_members
0,era5_truth_240,62,latitude,longitude,time,NaN,level,NaN,1959-01-01 00:00:00,2023-01-10 18:00:00,93544,13,NaN,NaN
1,graphcast_2020_240,14,latitude,longitude,time,prediction_timedelta,level,NaN,2019-11-16 00:00:00,2021-01-31 12:00:00,886,37,40.0,NaN
2,hres_0012_240,16,latitude,longitude,time,prediction_timedelta,level,NaN,2016-01-01 00:00:00,2022-12-31 12:00:00,5114,13,41.0,NaN
3,ifs_ens_240,15,latitude,longitude,time,prediction_timedelta,level,number,2018-01-01 00:00:00,2018-01-01 00:00:00,3652,3,61.0,50.0
4,neuralgcm_det_2020_240,9,latitude,longitude,time,prediction_timedelta,level,NaN,2020-01-01 00:00:00,2020-12-31 12:00:00,732,37,31.0,NaN



--- alignment_audit ---
/global/u2/s/suryact/Chaos_Theory/project1_code/flagship_predictability_nextpass/notebooks/outputs/audit/alignment_audit.csv


,dataset,variable_key,lead_hours,n_valid_times,forecast_shape,truth_shape,status
0,graphcast_2020_240,z500,24,180,180x240x121,180x240x121,ok
1,graphcast_2020_240,z500,72,176,176x240x121,176x240x121,ok
2,graphcast_2020_240,z500,120,172,172x240x121,172x240x121,ok
3,graphcast_2020_240,z500,168,168,168x240x121,168x240x121,ok
4,graphcast_2020_240,t850,24,180,180x240x121,180x240x121,ok
5,graphcast_2020_240,t850,72,176,176x240x121,176x240x121,ok
6,graphcast_2020_240,t850,120,172,172x240x121,172x240x121,ok
7,graphcast_2020_240,t850,168,168,168x240x121,168x240x121,ok
8,graphcast_2020_240,u850,24,180,180x240x121,180x240x121,ok
9,graphcast_2020_240,u850,72,176,176x240x121,176x240x121,ok
